In [20]:
import pandas as pd
import numpy as np
from ta import add_all_ta_features 
import warnings
warnings.filterwarnings('ignore')

# "ADD_ALL_TA_FEATURES is used for automated feature engineering on financial dataframes. Instead of calculating metrics one by one, 
# calling this single function automatically computes over 40+ technical indicators (generating over 80+ new data columns) 
# and appends them directly to your dataset.

# What it actually doesIt scans your standard OHLCV data (Open, High, Low, Close, Volume) 
# and creates features categorized across major trading dimensions:Volume Indicators: Accumulation/Distribution Index (ADI), 
# On-Balance Volume (OBV), Chaikin Money Flow (CMF).

# Volatility Indicators: Average True Range (ATR), Bollinger Bands,Keltner Channels.

# Trend Indicators: MACD, Average Directional Index (ADX), Ichimoku Kinko Hyo, Exponential Moving Averages (EMA).

# Momentum Indicators: Relative Strength Index (RSI), Stochastic Oscillator, Williams %R.


In [26]:
tickers=['RELIANCE.NS','INFY.NS','TCS.NS']

raw={}
for ticker in tickers:
    fname=ticker.replace(".","_")
    raw[ticker]=pd.read_csv(f"{fname}_raw.csv")

raw



{'RELIANCE.NS':             Close         High          Low         Open    Volume
 0      496.196381   498.985034   491.371654   498.077624   9746670
 1      489.733917   498.852251   487.343655   493.319266  15628818
 2      483.691925   493.363540   482.518940   490.220816  16288287
 3      486.303436   488.870748   478.535167   485.750155  18516544
 4      489.003510   495.067644   487.343608   489.999444  12060290
 ...           ...          ...          ...          ...       ...
 1230  1267.243042  1275.073539  1258.646635  1264.550536  16541784
 1231  1273.665527  1280.532666  1266.081931  1268.700379   7465664
 1232  1278.013184  1284.460338  1271.220103  1275.617031   9204156
 1233  1287.251709  1290.438257  1278.013144  1279.470544  12302636
 1234  1277.074341  1291.426291  1274.283051  1289.993617  10864584
 
 [1235 rows x 5 columns],
 'INFY.NS':             Close         High          Low         Open   Volume
 0      541.725830   542.744035   532.847111   538.386137  2943

In [30]:
def build_features(df):
    data=df.copy()
    
    data=add_all_ta_features(data, open="Open", high="High",low="Low", close="Close", volume="Volume",fillna=True)
    cols_to_keep = [
        'Close',
        'trend_sma_fast', 'trend_sma_slow', 'trend_ema_fast',  # Trend
        'trend_macd', 'trend_macd_signal', 'trend_macd_diff',  # MACD
        'momentum_rsi',                                          # RSI
        'volatility_bbh', 'volatility_bbl', 'volatility_bbw',  # Bollinger Bands
        'volatility_atr',                                        # ATR
        'volume_obv',                                            # Volume
    ]
    data = data[cols_to_keep]

    # Price derived features
    data['return_1d']  = data['Close'].pct_change(1)
    data['return_5d']  = data['Close'].pct_change(5)
    data['return_10d'] = data['Close'].pct_change(10)
    
    #Target variable-if tomorrows price is greater than today
    data['Target']=(data['Close'].shift(-1)>data['Close']).astype(int)
    
    data.dropna(inplace=True)
    return data
    
    

In [36]:
featured = {}
for ticker, df in raw.items():
    featured[ticker] = build_features(df)
    fname = ticker.replace(".", "_")
    featured[ticker].to_csv(f"{fname}_features.csv",index=False)
    print(f"{ticker}: {featured[ticker].shape}")

RELIANCE.NS: (1225, 17)
INFY.NS: (1225, 17)
TCS.NS: (1225, 17)
